# Modelado — TP Grupo 10
LightGBM + Optuna (150 trials)

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import MinMaxScaler
from joblib import dump, load


## Rutas y parámetros

In [ ]:
def find_repo_root():
    p = Path.cwd()
    for _ in range(6):
        if (p / 'work' / 'cleaned' / 'train_clean.csv').exists():
            return p
        p = p.parent
    raise FileNotFoundError('No se encontro el root del repositorio')

BASE_DIR         = find_repo_root()
PATH_TRAIN_CLEAN = BASE_DIR / 'work' / 'cleaned' / 'train_clean.csv'
PATH_TEST_CLEAN  = BASE_DIR / 'work' / 'cleaned' / 'test_clean.csv'
PATH_TO_MODELS   = BASE_DIR / 'work' / 'models'
PATH_PREDS       = BASE_DIR / 'delfina' / 'mica' / 'predicciones_test.csv'

SEED    = 42
N_FOLDS = 5

FEATURES_SEL = [
    'age_x_MaturitySize', 'Age', 'PhotoAmt', 'health_score',
    'health_complete', 'Breed1', 'Sterilized',
    'Breed2', 'photo_video_score'
]

print(f'BASE_DIR: {BASE_DIR}')
print(f'Train:    {PATH_TRAIN_CLEAN}')
print(f'Test:     {PATH_TEST_CLEAN}')
assert PATH_TRAIN_CLEAN.exists(), f'No existe: {PATH_TRAIN_CLEAN}'
assert PATH_TEST_CLEAN.exists(),  f'No existe: {PATH_TEST_CLEAN}'
print('Rutas OK')

## Carga de datos

In [ ]:
train = pd.read_csv(PATH_TRAIN_CLEAN)
test  = pd.read_csv(PATH_TEST_CLEAN)
print(f'Train: {train.shape} | Test: {test.shape}')

## Feature Engineering

In [ ]:
DROP_COLS = ['AdoptionSpeed', 'PetID', 'RescuerID', 'Name', 'Description',
             'has_name', 'has_description']

def build_features(df, scaler=None, fit=True):
    drop = [c for c in DROP_COLS if c in df.columns]
    X = df.drop(columns=drop).copy()
    X['health_score'] = (
        (df['Vaccinated'] == 1).astype(int) +
        (df['Dewormed']   == 1).astype(int) +
        (df['Sterilized'] == 1).astype(int) +
        (df['Health']     == 1).astype(int)
    )
    X['health_complete']    = (X['health_score'] == 4).astype(int)
    X['age_x_MaturitySize'] = (df['Age'] / 12) * df['MaturitySize']
    if fit:
        scaler = MinMaxScaler()
        X['age_x_MaturitySize'] = scaler.fit_transform(X[['age_x_MaturitySize']])
    else:
        X['age_x_MaturitySize'] = scaler.transform(X[['age_x_MaturitySize']])
    X['photo_video_score'] = df['PhotoAmt'] + df['VideoAmt'] * 2
    return X[FEATURES_SEL], scaler

X_train, scaler = build_features(train, fit=True)
y_train = train['AdoptionSpeed']
X_test,  _      = build_features(test, scaler=scaler, fit=False)
y_test  = test['AdoptionSpeed']
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

## Baseline

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

base_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'verbosity': -1, 'random_state': SEED, 'class_weight': 'balanced',
}
kappa_folds = []
for fold, (idx_tr, idx_val) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[idx_tr], X_train.iloc[idx_val]
    y_tr, y_val = y_train.iloc[idx_tr], y_train.iloc[idx_val]
    m = lgb.LGBMClassifier(**base_params, n_estimators=500)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    k = cohen_kappa_score(y_val, m.predict(X_val), weights='quadratic')
    kappa_folds.append(k)
    print(f'  Fold {fold} — Kappa: {k:.4f}')
print(f'\nBaseline — Kappa medio: {np.mean(kappa_folds):.4f} ± {np.std(kappa_folds):.4f}')

## Optuna — 150 trials

In [ ]:
def objective(trial):
    params = {
        'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
        'verbosity': -1, 'random_state': SEED, 'class_weight': 'balanced',
        'n_estimators': 500,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 20, 300),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    kappas = []
    for idx_tr, idx_val in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[idx_tr], X_train.iloc[idx_val]
        y_tr, y_val = y_train.iloc[idx_tr], y_train.iloc[idx_val]
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        kappas.append(cohen_kappa_score(y_val, m.predict(X_val), weights='quadratic'))
    return np.mean(kappas)

db_path = str(BASE_DIR / 'work' / 'db.sqlite3').replace('\\', '/')
storage_url = f'sqlite:///{db_path}'

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    storage=storage_url,
    study_name='lgbm_kappa_tp10',
    load_if_exists=True
)
study.optimize(objective, n_trials=150, show_progress_bar=True)

print(f'\nMejor Kappa: {study.best_value:.4f}')
print(f'Mejores params: {study.best_params}')

## Modelo final

In [ ]:
best_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'verbosity': -1, 'random_state': SEED, 'n_estimators': 1000,
    'class_weight': 'balanced',
    **study.best_params
}
X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
    X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train
)
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_tr_f, y_tr_f, eval_set=[(X_val_f, y_val_f)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
PATH_TO_MODELS.mkdir(parents=True, exist_ok=True)
dump(final_model, PATH_TO_MODELS / 'lgbm_final_tp10.joblib')
print('Modelo guardado.')

## Evaluación sobre test

In [ ]:
y_pred  = final_model.predict(X_test)
kappa   = cohen_kappa_score(y_test, y_pred, weights='quadratic')
acc     = accuracy_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print('=== Métricas sobre test ===')
print(f'  Cohen Kappa (quadratic): {kappa:.4f}')
print(f'  Accuracy:                {acc:.4f}')
print(f'  Balanced Accuracy:       {bal_acc:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'Pred {i}' for i in range(5)],
            yticklabels=[f'Real {i}' for i in range(5)])
plt.title('Matriz de confusión — LightGBM tuneado (150 trials)')
plt.ylabel('Clase real')
plt.xlabel('Clase predicha')
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    'feature':    FEATURES_SEL,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis')
plt.title('Importancia de features (gain)')
plt.tight_layout()
plt.show()

## Predicciones sobre test

In [ ]:
resultado = test[['PetID']].copy()
resultado['AdoptionSpeed_real']     = y_test.values
resultado['AdoptionSpeed_predicho'] = y_pred
PATH_PREDS.parent.mkdir(parents=True, exist_ok=True)
resultado.to_csv(PATH_PREDS, index=False)
print(f'Predicciones guardadas en: {PATH_PREDS}')
resultado.head(10)